# 香港特区`CBERS-04`卫星`MUX`影像辐射定标

## 程序包初始化

### 内建程序包

In [1]:
import io, sys, os; 
try:
    nb_dir = nb_dir; 
except NameError: 
    nb_dir = os.getcwd(); 
    sys.path.append(nb_dir); 

In [2]:
import re; 
import collections as coll, itertools as it; 
from __future__ import print_function; 

In [3]:
from datetime import datetime; 

In [4]:
import sqlite3; 

### 第三方和自定义程序包

In [5]:
import numpy as np, pandas as pd; 

In [6]:
_ = sys.stdout; 
with io.BytesIO() as sys.stdout: import idlpy; 
sys.stdout = _; 

In [7]:
py_pkg_dir = os.path.normpath(
    os.path.join(nb_dir, os.pardir, os.pardir, "Python_Package")
); 
sys.path.append(py_pkg_dir); 

In [8]:
import cresda_metadata_parse as cresda; 
cresda = cresda.importlib.reload(cresda); 

In [9]:
idl_pkg_dir = os.path.normpath(
    os.path.join(nb_dir, os.pardir, os.pardir, "IDL_Package")
); 

In [10]:
import cresda_metadata_parse as cresda; 
cresda = cresda.importlib.reload(cresda); 

In [11]:
idl_pkg_dir = os.path.normpath(
    os.path.join(nb_dir, os.pardir, os.pardir, "IDL_Package")
); 

In [12]:
idl_cb04 = os.path.normpath(
    os.path.join(idl_pkg_dir, "CB04_MUX_Preproc")
); 
idl_cb04_subpkg = (
    "gain_bias", 
); 
idl_cb04_cmpl_seq = tuple(
    os.path.join(idl_cb04, pkg + ".pro")
    for pkg in idl_cb04_subpkg
); 

In [13]:
idlpy.IDL.e = idlpy.IDL.envi(headless=False); 

% Restored file: ENVI.
% Loaded DLM: HPGRAPHICS.
% Compiled module: ENVI_VECTOR_MASK_RASTER_CLASSIC.
% Loaded DLM: PNG.
% Loaded DLM: URL.


In [14]:
for pkg in idl_cb04_cmpl_seq: 
    idlpy.IDL.run(".compile -v {pkg}".format(pkg=pkg)); 

## 辐射定标系数指定
背景: 
* 大部分年份`CB04 MUX`L2级光学影像元数据`xml`中的辐射定标系数有误
* `ENVI`第三方工具"中国国产卫星支持工具"错误读取上述数据, 所得定标结果错误, 部分年份辐亮度计算结果偏低, 进一步导致`FLAASH`大气校正所得反射率结果出现大量负值. 

In [15]:
cb04_mux_gain = {
    2015: np.array([0.5274, 0.6609, 0.5998, 0.4900]), 
    2016: np.array([0.5728, 0.6609, 0.6799, 0.5514]), 
    2017: np.array([0.5494, 0.7915, 0.6571, 0.5063]), 
    2018: np.array([0.6141, 0.8874, 0.7288, 0.5550]), 
    2019: np.array([0.5453, 0.9615, 0.6474, 0.3970]), 
    2024: np.array([1.6945, 1.5950, 1.5400, 1.3624]), 
}

In [16]:
#判定遥感图像适用哪一年的定标系数
#某一年定标系数的适用范围: 当年公历2月1日 (含) 起, 至次年公历1月31日 (含) 止. 
def acq_date_to_calib_year(date): 
    #由于起始日期是当年2月1日, 无论是平年还是闰年, 均为当年元旦后31日, 
    #因此直接将日期减去31天后, 取年份
    date_modif = np.int64(date).astype(np.dtype("datetime64[ms]")); 
    date_modif -= np.timedelta64(31, "D"); 
    date_modif = datetime.fromtimestamp(date_modif.astype(np.int64) / 1000); 
    return date_modif.year; 

In [17]:
rs_meta_items_sqlite = sqlite3.connect(os.path.join(
    nb_dir, os.pardir, "RS_Parameter_Calc", "CBERS-04_metadata.sqlite"
) ); 

In [18]:
acq_date = pd.read_sql(
    """	
    SELECT 
        filename, scene_date
	From info_metadata_cb04
    """, con=rs_meta_items_sqlite
); 
acq_date.set_index("filename", inplace=True); 

In [19]:
rs_meta_dir = os.path.normpath(
    os.path.join(
        nb_dir, os.pardir, os.pardir, 
        os.pardir, "Source", "Imagery"
    )
); 

In [20]:
mux_finder = cresda.cb04.MUX(); 
mux_finder.source_dir = rs_meta_dir; 
mux_finder.target_dir = rs_meta_dir; 

In [21]:
cb04_mux_scenes = tuple(scene for scene in mux_finder.traverse()); 

In [22]:
for scene in cb04_mux_scenes: 
    hdr_name = scene.groupdict["archive"] + ".hdr"; 
    if os.path.isfile(hdr_name): 
        continue; 
    #读取元数据xml文件位置
    xml_path = scene.target[0]; 
    #查找影像对应的定标系数, 注意各波段bias恒为0. 
    tif_name = scene.groupdict["archive"] + ".TIF"; 
    raster_acq_date = acq_date.scene_date[tif_name]; 
    raster_calib_year = acq_date_to_calib_year(raster_acq_date); 
    raster_rad_gain = cb04_mux_gain[raster_calib_year]; 
    raster_rad_bias = np.zeros_like(raster_rad_gain); 
    #注意: 尽量不要直接通过python的idlpy session调用EnviOpenChinaRaster
    #已知的问题: 
    #ENVI插件"中国卫星支持工具"的IDL接口EnviOpenChinaRaster在idlpy反复
    #调用多次会卡死, 但是直接在IDL的pro中调用多次不会卡死, 机制尚未探明. 
    idlpy.IDL.cb04_mux_metadata_acq(
        xml_path, raster_rad_gain, raster_rad_bias
    ); 
    #显示进度
    print("Product {arx} processed at {time}".format(
        arx=scene.groupdict["archive"], 
        time=datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    ), end="\r"); 
    sys.stdout.flush(); 
print("{0: <79}".format("All products have been processed. ")); 
sys.stdout.flush(); 

% Loaded DLM: NATIVE.
% Loaded DLM: JPEG2000.
% Loaded DLM: JPEG.
% Loaded DLM: HDF5.
% Loaded DLM: MAP_PE.
All products have been processed.                                              


## 辐射定标
* 使用`ENVITask`执行辐射定标, 确定辐射定标配置参数, 使定标结果可直接用于后续`FLAASH`大气校正; 
* `ENVIRadiometricCalibrationTask`可指定辐射定标类型, 输出数据类型和缩放倍率; 
* `ENVIRadiometricCalibrationTask`不可指定输出栅格数据组织形式, 将与输入数据保持一致 (`BIP`)

In [23]:
rad_calib_for_flaash = idlpy.IDL.EnviTask("RadiometricCalibration"); 
rad_calib_for_flaash.calibration_type = "Radiance"; 
rad_calib_for_flaash.output_data_type = "Float"; 
rad_calib_for_flaash.scale_factor = 0.10; 

In [24]:
for scene in cb04_mux_scenes: 
    #指定输入和输出路径
    output_uri = os.path.join(
        nb_dir, scene.groupdict["archive"] + "_RAD.dat"
    ); 
    if os.path.isfile(output_uri): 
        continue; 
    input_uri = scene.source; 
    #在ENVI中打开文件
    raster_dn = idlpy.IDL.e.openraster(input_uri); 
    #指定辐射定标任务的输入和输出, 并执行
    rad_calib_for_flaash.input_raster = raster_dn; 
    rad_calib_for_flaash.output_raster_uri = output_uri; 
    rad_calib_for_flaash.execute(); 
    #关闭辐射定标输入和输出结果
    raster_rad = rad_calib_for_flaash.output_raster; 
    raster_dn.close(); 
    raster_rad.close(); 
    #显示进度
    print("Product {arx} processed at {time}".format(
        arx=scene.groupdict["archive"], 
        time=datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    ), end="\r"); 
    sys.stdout.flush(); 
print("{0: <79}".format("All products have been processed. ")); 
sys.stdout.flush(); 

All products have been processed.                                              
